TASK 1.2

In [0]:
#dont run this !!!!
import pandas as pd

course_path = "/Volumes/edtech_project/edtech_bronze/edtech_project/course_catalog/course_catalog.csv"

# Read first 251 rows with Pandas
pdf = pd.read_csv(course_path, nrows=251)

# Convert to Spark DataFrame
course_df = spark.createDataFrame(pdf)

# Save as Delta
(course_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("course_catalog")
)

print(f"Total valid rows saved: {course_df.count()}")

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
#enrollments 
enroll_path = "/Volumes/edtech_project/edtech_bronze/edtech_project/enrollments/"

# 1. Read daily delta file
enroll_df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(enroll_path)
)
# 2. Metadata
enroll_df = (enroll_df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_load_ts", current_timestamp())
    .withColumn("_schema_version", lit("v1"))
)

# 3. Referential Integrity (join with course_catalog)
course_df = spark.table("course_catalog")

enroll_df = enroll_df.join(
    course_df,
    on="course_id",
    how="inner"   # keeps only valid course_ids
)

# 4. Create temp view for merge
enroll_df.createOrReplaceTempView("enrollments_temp")

In [0]:
#target table
(enroll_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("enrollments_bronze")
)



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("enrollment_id").orderBy(col("_load_ts").desc())
enroll_df = enroll_df.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop("rn")

In [0]:
%sql
MERGE INTO enrollments_bronze AS target
USING enrollments_temp AS source
ON target.enrollment_id = source.enrollment_id

WHEN MATCHED THEN 
  UPDATE SET
    target.student_id = source.student_id,
    target.course_id = source.course_id,
    target.enrolled_date = source.enrolled_date,
    target.expected_completion_date = source.expected_completion_date,
    target.current_progress_pct = source.current_progress_pct,
    target.last_accessed_date = source.last_accessed_date,
    target.status = source.status,
    target._source_file = source._source_file,
    target._load_ts = source._load_ts,
    target._schema_version = source._schema_version

WHEN NOT MATCHED THEN 
  INSERT *

In [0]:
%sql
-- display the table
SELECT * 
FROM enrollments_bronze
LIMIT 5;

In [0]:
%sql
select count(*) from enrollments_bronze;